# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [1]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/jjay321-oss/2026-HYU-AUE8088-PA2

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 90 (delta 36), reused 7 (delta 7), pack-reused 33 (from 2)
Receiving objects: 100% (90/90), 1.96 MiB | 5.99 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 

In [2]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.models.resnet import resnet18
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Mounted at /content/drive


In [3]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
STRATEGY_NAME = "rare-hard-combo"   # 본인 전략명 (Run 이름에 들어감)
WANDB_TAGS    = ["level5", STRATEGY_NAME]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jjay321 (jjay321-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=e69f90ec-d326-4ee3-96e6-5ae6a144e1c9
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:02<00:00, 88.6MB/s]


압축 해제 중...
완료 → ../data/set_a


In [5]:
# 1단계 — best base 모델로 Set B 의 모든 이미지를 score.
model = resnet18().to(device)
#model.load_state_dict(torch.load("../checkpoints/level3_best.pth", map_location=device)["state_dict"])

ckpt_path = "/content/drive/MyDrive/level3_focal_weather_sampler1.pth"
ckpt = torch.load(ckpt_path, map_location=device)

if isinstance(ckpt, dict) and "state_dict" in ckpt:
    ckpt = ckpt["state_dict"]

model.load_state_dict(ckpt)
model.eval()

set_b = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)

preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)

# 이미지별 불확실성 (uncertainty) 계산: 1 - max-softmax 를 3개 head 평균.
max_probs = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)
print(f"unc shape: {uncertainty.shape}, mean={uncertainty.mean():.3f}")

unc shape: (15000,), mean=0.200


In [8]:
# 2단계 — 본인의 선별 알고리즘을 설계하세요.
#
# Set B 의 정답 라벨은 set_b 의 BDDAttrDataset (split="mining") 에 이미 채워져 있습니다.
# preds_b (모델 예측) 와 set_b 의 sample.weather/scene/timeofday (정답) 를 함께 활용 가능.
#
# 아래는 placeholder (top-1000 most-uncertain). 본인 구현으로 교체하세요.
# Tip: 단일 신호보다 여러 신호를 결합하는 편이 보통 더 좋은 성능을 냅니다. 예시:
#   score_i = lam * uncertainty_i  +  (1 - lam) * is_rare_class_i
# 2단계 — 본인의 선별 알고리즘 설계
import json
import numpy as np

K = 1000

# Set A train 분포 확인용
train_ds = BDDAttrDataset("../data/set_a", "train", transform=eval_transform())

# Set B 정답 라벨 가져오기
labels_b = {}
labels_b["weather"] = np.array([s.weather for s in set_b.samples])
labels_b["scene"] = np.array([s.scene for s in set_b.samples])
labels_b["timeofday"] = np.array([s.timeofday for s in set_b.samples])

# 1) Rare-class score
# Set A에서 적게 나온 class일수록 높은 점수
rare_score = np.zeros(len(set_b), dtype=np.float32)

for a in ATTRIBUTES:
    counts = train_ds.class_counts(a)
    counts = np.array(counts, dtype=np.float32)

    inv = 1.0 / np.sqrt(counts + 1.0)
    inv = inv / inv.max()

    rare_score += inv[labels_b[a]]

rare_score = rare_score / len(ATTRIBUTES)

# 2) Hard-example score
# base model이 틀린 속성이 많을수록 높은 점수
error_score = np.zeros(len(set_b), dtype=np.float32)

for a in ATTRIBUTES:
    error_score += (preds_b[a] != labels_b[a]).astype(np.float32)

error_score = error_score / len(ATTRIBUTES)

# 3) 특정 소수 클래스 bonus
bonus = np.zeros(len(set_b), dtype=np.float32)

def find_class(attr, names):
    cls = [c.lower().replace("_", "-").replace("/", "-") for c in CLASS_NAMES[attr]]
    for name in names:
        name = name.lower().replace("_", "-").replace("/", "-")
        if name in cls:
            return cls.index(name)
    return None

snowy_idx = find_class("weather", ["snowy"])
foggy_idx = find_class("weather", ["foggy"])
rainy_idx = find_class("weather", ["rainy"])
dawn_idx = find_class("timeofday", ["dawn-dusk", "dawn/dusk"])

if snowy_idx is not None:
    bonus += (labels_b["weather"] == snowy_idx) * 0.20

if foggy_idx is not None:
    bonus += (labels_b["weather"] == foggy_idx) * 0.20

if dawn_idx is not None:
    bonus += (labels_b["timeofday"] == dawn_idx) * 0.15

if rainy_idx is not None and dawn_idx is not None:
    bonus += ((labels_b["weather"] == rainy_idx) & (labels_b["timeofday"] == dawn_idx)) * 0.10

# 4) 최종 score 계산
# uncertainty는 1단계에서 이미 계산된 값
score = 0.45 * rare_score + 0.35 * uncertainty + 0.20 * error_score + bonus

# 5) score가 높은 상위 1000장 선택
order = np.argsort(-score)[:K]

picks = []
for i in order:
    s = set_b.samples[i]

    picks.append({
        "image_id": s.image_id,
        "weather": int(s.weather),
        "scene": int(s.scene),
        "timeofday": int(s.timeofday),
        "score": float(score[i]),
        "uncertainty": float(uncertainty[i]),
        "rare_score": float(rare_score[i]),
        "error_score": float(error_score[i]),
        "reason": "rare class + uncertainty + prediction error",
    })

with open("../level5_picks.json", "w") as f:
    json.dump({
        "strategy": (
            "Set A에서 부족한 class를 보강하기 위해 rare-class score를 사용"
            "base model의 confidence가 낮은 uncertainty sample과 실제 라벨 대비 오분류된 hard example을 함께 고려"
            "추가로 snowy, foggy, dawn/dusk와 같은 소수 클래스에 bonus를 부여하여 최종 score가 높은 상위 1000장을 선택"
        ),
        "num_picks": len(picks),
        "picks": picks,
    }, f, indent=2)

print(f"saved {len(picks)} picks")
print("score mean:", score.mean(), "score max:", score.max())

saved 1000 picks
score mean: 0.31280813 score max: 0.96946764


In [ ]:
# 3단계 — Set A + 본인이 고른 picks 로 재학습. 학습 메트릭은 wandb 로 자동 로깅.
extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in picks]
train_aug = BDDAttrDataset("../data/set_a", "train", transform=train_transform(), extra_picks=extra)
val_ds    = BDDAttrDataset("../data/set_a", "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
loader_tr  = DataLoader(train_aug, batch_size=64, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
loader_val = DataLoader(val_ds,    batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

set_seed(SEED, deterministic=True)
model2 = resnet18().to(device)
optim  = torch.optim.AdamW(model2.parameters(), lr=3e-4, weight_decay=5e-4)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=25)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level5-{STRATEGY_NAME}",
    config={
        "backbone": "resnet18", "strategy": STRATEGY_NAME,
        "num_picks": len(picks), "epochs": 30, "batch": 64, "lr": 3e-4, "seed": SEED,
    },
    tags=WANDB_TAGS,
)
trainer = MultiTaskTrainer(model2, optim, sched, losses, device, TrainConfig(epochs=25), logger=logger)
history = trainer.fit(loader_tr, loader_val)

# 학습 종료 후 — 속성별 confusion matrix 와 picks 분포를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model2, loader_val, device)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])

# 본인이 고른 1,000장의 (예측) 분포 — Curation 의도가 의도대로 반영됐는지 검증
from collections import Counter
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    rows = [[CLASS_NAMES[a][k], cnt.get(k, 0)] for k in range(NUM_CLASSES[a])]
    logger.log_table(f"picks/distribution_{a}", ["class", "count"], rows)

os.makedirs("../checkpoints", exist_ok=True)
torch.save({"state_dict": model2.state_dict(), "history": history},
           "../checkpoints/level5_final.pth")
os.makedirs("/content/drive/MyDrive/pa2_checkpoints", exist_ok=True)

torch.save(
    {"state_dict": model2.state_dict(), "history": history},
    "/content/drive/MyDrive/pa2_checkpoints/level5_final.pth"
)
!cp ../level5_picks.json "/content/drive/MyDrive/pa2_checkpoints/level5_picks.json"
logger.finish()

epoch,▁
lr,▁
train/loss,▁
val/avg_macro_f1,▁
val/mf1_scene,▁
val/mf1_timeofday,▁
val/mf1_weather,▁
epoch,1
lr,0.0003
train/loss,2.41898
val/avg_macro_f1,0.42201


[epoch 01/25] train_loss=2.4190  val_avg_MF1=0.4220  per={'weather': 0.2853372572188994, 'scene': 0.27363428222260383, 'timeofday': 0.7070591112971892}


train e2:   5%|▌         | 5/94 [00:01<00:33,  2.62it/s, loss=2.2345]

In [ ]:
# 4단계 — Kaggle 제출용 CSV 생성.
test_ds = BDDAttrDataset("../data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
write_submission("../submission/level5_submission.csv", ids_te, preds_te)
print("submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.")

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.